<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/Webscraping_News.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install feedparser beautifulsoup4 openai

import feedparser, smtplib, getpass
from bs4 import BeautifulSoup
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from openai import OpenAI
from google.colab import userdata

# =========================
# 🔐 OPENAI SECRET
# =========================
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Add it in Colab Secrets.")

client = OpenAI(api_key=OPENAI_API_KEY)

# =========================
# 🧪 MODE SELECTION
# =========================
mode = input("Choose mode (demo / real): ").strip().lower()

# =========================
# ✍️ EMAIL INPUT + VALIDATION
# =========================
def get_email_details():
    print("\nEMAIL SETUP:")
    print("- Use real Gmail + App Password for REAL mode")
    print("- Use anything for DEMO mode\n")

    sender = input("Enter sender Gmail: ").strip()
    password = getpass.getpass("Enter Gmail App Password: ").replace(" ", "").strip()
    receiver = input("Enter receiver email: ").strip()

    if not sender or not receiver:
        raise ValueError("Email fields cannot be empty.")

    if "@" not in sender or "@" not in receiver:
        raise ValueError("Invalid email format.")

    return sender, password, receiver

sender_email, gmail_app_password, receiver_email = get_email_details()

# =========================
# NEWS SOURCES
# =========================
NEWS_FEEDS = {
    "Google News AI": "https://news.google.com/rss/search?q=artificial+intelligence",
    "Google News GenAI": "https://news.google.com/rss/search?q=generative+ai",
    "BBC": "https://feeds.bbci.co.uk/news/rss.xml"
}

# =========================
# FETCH NEWS
# =========================
def fetch_news(n=5):
    articles = []
    for source, url in NEWS_FEEDS.items():
        feed = feedparser.parse(url)
        for entry in feed.entries[:n]:
            summary = BeautifulSoup(entry.get("summary",""), "html.parser").get_text()
            articles.append({
                "source": source,
                "title": entry.get("title",""),
                "summary": summary,
                "link": entry.get("link","")
            })
    return articles

# =========================
# 🤖 AI AGENT
# =========================
def ai_news_agent(articles):
    text = ""
    for i,a in enumerate(articles,1):
        text += f"\n{i}. {a['title']}\n{a['summary']}\n{a['link']}\n"

    prompt = f"""
Create a clean daily AI news digest.

Rules:
- Pick top 8 useful news
- Each item:
  Headline
  2-line summary
  Why it matters
  Link
- End with "Trend of the Day"

News:
{text}
"""

    res = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )
    return res.output_text

# =========================
# 📧 SEND EMAIL / DEMO
# =========================
def send_email(sender, pwd, receiver, subject, body, mode):

    if mode == "demo":
        print("\n🧪 DEMO MODE (No email sent)")
        print("To:", receiver)
        print("Subject:", subject)
        print("\n--- EMAIL CONTENT ---\n")
        print(body[:1200], "\n...")
        return

    try:
        msg = MIMEMultipart()
        msg["From"] = sender
        msg["To"] = receiver
        msg["Subject"] = subject
        msg.attach(MIMEText(body, "plain"))

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(sender, pwd)
            server.send_message(msg)

        print("✅ Email sent successfully!")

    except smtplib.SMTPAuthenticationError:
        print("\n❌ Login failed.")
        print("Use Gmail App Password, not your normal password.")

    except Exception as e:
        print("\n❌ Email sending failed:", e)
        print("Tip: Use DEMO mode if testing.")

# =========================
# 🚀 RUN PROJECT
# =========================
articles = fetch_news(5)
digest = ai_news_agent(articles)

today = datetime.now().strftime("%d %B %Y")
subject = f"Daily GenAI News - {today}"

print("\n📄 PREVIEW:\n")
print(digest)

send_email(sender_email, gmail_app_password, receiver_email, subject, digest, mode)